# Register Models in LiteLLM

This notebook registers your deployed LLM and embedding services in LiteLLM, creating a unified API gateway.

**Prerequisites**:
- `00-platform-validation.ipynb` completed successfully
- Chat LLM deployed via `tkt-tensorrt-llm` (e.g., gpt-oss-20b)
- Embedding service deployed via `tkt-text-embeddings` (e.g., nomic-embed-text-v1.5)

**After this notebook**:
- Both models accessible via LiteLLM's unified OpenAI-compatible API
- Single endpoint for chat completions and embeddings

In [ ]:
import os
import requests
from IPython.display import display, HTML

def success(msg):
    display(HTML(f'<span style="color: #00c896; font-weight: bold;">✓ {msg}</span>'))

def error(msg):
    display(HTML(f'<span style="color: #e74c3c; font-weight: bold;">✗ {msg}</span>'))

def info(msg):
    display(HTML(f'<span style="color: #3498db;">ℹ {msg}</span>'))

## Configuration

Set your deployed service endpoints. Update these to match your deployment names.

In [ ]:
# LiteLLM configuration
LITELLM_ENDPOINT = os.environ.get('LITELLM_ENDPOINT')
LITELLM_MASTER_KEY = os.environ.get('LITELLM_MASTER_KEY')

# Your deployed services - UPDATE THESE to match your deployment
# Format: http://{service-name}.{namespace}.svc:{port}
LLM_SERVICE_URL = "http://gpt-oss.default.svc:8355"  # Your tkt-tensorrt-llm deployment
LLM_MODEL_NAME = "gpt-oss"  # Name to register in LiteLLM

EMBEDDING_SERVICE_URL = "http://nomic-embed.default.svc:8355"  # Your tkt-text-embeddings deployment
EMBEDDING_MODEL_NAME = "nomic-embed"  # Name to register in LiteLLM

print(f"LiteLLM Endpoint: {LITELLM_ENDPOINT}")
print(f"LLM Service: {LLM_SERVICE_URL}")
print(f"Embedding Service: {EMBEDDING_SERVICE_URL}")

---
## 1. Verify Services Are Running

Check that your deployed services are healthy before registering.

In [ ]:
# Check LLM service health
try:
    resp = requests.get(f"{LLM_SERVICE_URL}/health", timeout=10)
    if resp.status_code == 200:
        success(f"LLM service healthy: {LLM_SERVICE_URL}")
        info(f"Response: {resp.json()}")
    else:
        error(f"LLM service returned {resp.status_code}")
except Exception as e:
    error(f"Cannot reach LLM service: {e}")
    info("Make sure your tkt-tensorrt-llm deployment is running")

In [ ]:
# Check Embedding service health
try:
    resp = requests.get(f"{EMBEDDING_SERVICE_URL}/health", timeout=10)
    if resp.status_code == 200:
        success(f"Embedding service healthy: {EMBEDDING_SERVICE_URL}")
        info(f"Response: {resp.json()}")
    else:
        error(f"Embedding service returned {resp.status_code}")
except Exception as e:
    error(f"Cannot reach Embedding service: {e}")
    info("Make sure your tkt-text-embeddings deployment is running")

---
## 2. Register LLM in LiteLLM

Register your chat/completion model via LiteLLM's `/model/new` API.

In [ ]:
# Register LLM model
llm_config = {
    "model_name": LLM_MODEL_NAME,
    "litellm_params": {
        "model": f"openai/{LLM_MODEL_NAME}",
        "api_base": f"{LLM_SERVICE_URL}/v1",
        "api_key": "not-needed"  # Local service, no API key required
    },
    "model_info": {
        "description": "Local GPT-OSS 20B via TensorRT-LLM",
        "mode": "chat"
    }
}

try:
    resp = requests.post(
        f"{LITELLM_ENDPOINT}/model/new",
        headers={
            "Authorization": f"Bearer {LITELLM_MASTER_KEY}",
            "Content-Type": "application/json"
        },
        json=llm_config,
        timeout=30
    )
    
    if resp.status_code == 200:
        success(f"Registered LLM model: {LLM_MODEL_NAME}")
        info(f"Response: {resp.json()}")
    elif resp.status_code == 400 and "already exists" in resp.text.lower():
        info(f"Model {LLM_MODEL_NAME} already registered")
    else:
        error(f"Failed to register LLM: {resp.status_code}")
        info(f"Response: {resp.text}")
except Exception as e:
    error(f"Error registering LLM: {e}")

---
## 3. Register Embedding Model in LiteLLM

Register your embedding model.

In [ ]:
# Register Embedding model
embedding_config = {
    "model_name": EMBEDDING_MODEL_NAME,
    "litellm_params": {
        "model": f"openai/{EMBEDDING_MODEL_NAME}",
        "api_base": f"{EMBEDDING_SERVICE_URL}/v1",
        "api_key": "not-needed"
    },
    "model_info": {
        "description": "Local nomic-embed-text-v1.5 embeddings",
        "mode": "embedding"
    }
}

try:
    resp = requests.post(
        f"{LITELLM_ENDPOINT}/model/new",
        headers={
            "Authorization": f"Bearer {LITELLM_MASTER_KEY}",
            "Content-Type": "application/json"
        },
        json=embedding_config,
        timeout=30
    )
    
    if resp.status_code == 200:
        success(f"Registered Embedding model: {EMBEDDING_MODEL_NAME}")
        info(f"Response: {resp.json()}")
    elif resp.status_code == 400 and "already exists" in resp.text.lower():
        info(f"Model {EMBEDDING_MODEL_NAME} already registered")
    else:
        error(f"Failed to register Embedding: {resp.status_code}")
        info(f"Response: {resp.text}")
except Exception as e:
    error(f"Error registering Embedding: {e}")

---
## 4. Verify Registration

List models registered in LiteLLM.

In [ ]:
# List registered models
try:
    resp = requests.get(
        f"{LITELLM_ENDPOINT}/model/info",
        headers={"Authorization": f"Bearer {LITELLM_MASTER_KEY}"},
        timeout=30
    )
    
    if resp.status_code == 200:
        models = resp.json()
        success(f"Found {len(models.get('data', []))} registered models")
        for model in models.get('data', []):
            model_name = model.get('model_name', 'unknown')
            mode = model.get('model_info', {}).get('mode', 'unknown')
            info(f"  - {model_name} ({mode})")
    else:
        error(f"Failed to list models: {resp.status_code}")
except Exception as e:
    error(f"Error listing models: {e}")

---
## 5. Test via LiteLLM Gateway

Test that both models work through the unified LiteLLM API.

In [ ]:
from openai import OpenAI

# Create client pointing to LiteLLM
client = OpenAI(
    base_url=LITELLM_ENDPOINT,
    api_key=LITELLM_MASTER_KEY
)

info(f"OpenAI client configured for LiteLLM at {LITELLM_ENDPOINT}")

In [ ]:
# Test Chat Completion
try:
    response = client.chat.completions.create(
        model=LLM_MODEL_NAME,
        messages=[{"role": "user", "content": "Say 'Hello from Thinkube!' in exactly those words."}],
        max_tokens=20
    )
    
    success(f"Chat completion via LiteLLM works!")
    info(f"Model: {LLM_MODEL_NAME}")
    info(f"Response: {response.choices[0].message.content}")
except Exception as e:
    error(f"Chat completion failed: {e}")

In [ ]:
# Test Embeddings
try:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL_NAME,
        input="This is a test sentence for embedding."
    )
    
    embedding = response.data[0].embedding
    success(f"Embeddings via LiteLLM works!")
    info(f"Model: {EMBEDDING_MODEL_NAME}")
    info(f"Embedding dimensions: {len(embedding)}")
    info(f"First 5 values: {embedding[:5]}")
except Exception as e:
    error(f"Embeddings failed: {e}")

---
## Summary

You now have a unified API gateway via LiteLLM:

```python
from openai import OpenAI

client = OpenAI(
    base_url=os.environ['LITELLM_ENDPOINT'],
    api_key=os.environ['LITELLM_MASTER_KEY']
)

# Chat
client.chat.completions.create(model="gpt-oss", messages=[...])

# Embeddings  
client.embeddings.create(model="nomic-embed", input="text")
```

**Benefits**:
- Single endpoint for all AI capabilities
- OpenAI-compatible API
- Cost tracking via LiteLLM
- Easy to swap models later

---

**Next**: Continue to `research-assistant/02-langchain-rag.ipynb` to build the RAG pipeline.